# 08 · Sentence Embeddings & Semantic Search in PyTorch

| | |
|---|---|
| **Author** | Chandan C |
| **GitHub** | [github.com/chandanc5525](https://github.com/chandanc5525) |
| **Course** | Agentic AI BootCamp · Module 02 – Advanced PyTorch |
| **Repository** | [AgenticAI_BootCamp-/02_advanced_pytorch](https://github.com/chandanc5525/AgenticAI_BootCamp-/tree/main/02_advanced_pytorch) |
| **Level** | Advanced |
| **Prerequisites** | Notebooks 04, 06 and 07 |

---

### What you will learn
- Contextual sentence encoder (Transformer + pooling) built with `nn.TransformerEncoder`
- Pooling strategies: mean, max, CLS - with correct padding masks
- Contrastive training: InfoNCE, supervised contrastive, batch-hard triplet, Matryoshka
- Retrieval metrics: Hit@k, Recall@k, MRR, nDCG@k - and a TF-IDF baseline
- Vector search: exact top-k, IVF (k-means) approximate search, int8 and binary quantisation, re-scoring
- Chunking, an `EmbeddingModel.encode` API, saving / loading an index
- Using a pretrained Hugging Face encoder with the same code path

> Every cell runs on a plain CPU. If a GPU (CUDA / Apple MPS) is present, the notebooks use it automatically.

## 0. Setup

In [ ]:
import math, random, re, time, collections, os, tempfile, itertools
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

def seed_everything(s=0):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_everything()
device = torch.device("cpu")          # tiny models: CPU is plenty (and keeps results reproducible)
print(torch.__version__)

## 1. From word vectors to sentence vectors
Static word embeddings (notebook 07) give *one* vector per word. A **sentence encoder** produces a vector for a whole text **in context**:

```
 text ─► tokens ─► token embeddings (+ positions) ─► Transformer layers ─► pool over tokens ─► (project) ─► L2-normalise ─► vector
```
Cosine similarity between two such vectors then measures semantic similarity, which is the basis of semantic search, RAG retrieval, clustering and deduplication.

## 2. A synthetic labelled corpus
Real paraphrase data is large, so to stay offline we **generate** it. Eight topics each have word pools (subjects, verbs, objects, places) and a few sentence frames. Two sentences about the same topic are *semantically related but usually share few or no content words* ("the chef roasted the soup" vs "a baker simmered some sauce") - exactly the situation where keyword search fails and embeddings help.

| Split | Size | Role |
|---|---|---|
| **train** | 80 sentences / topic | train the encoder (labels = topic) |
| **docs** | 6 sentences / topic | the searchable collection (never seen in training) |
| **queries** | 5 sentences / topic | test queries; relevant = docs of the same topic |

> The data is synthetic and deliberately easy; absolute numbers will be higher than on real text. What matters is the *workflow*, which is identical for real data (see section 14 for a pretrained model).

In [ ]:
POOLS = {
 "cooking":  dict(S="chef cook baker grandmother waiter butcher caterer apprentice".split(), V="baked roasted boiled grilled simmered chopped seasoned stirred".split(),
                  O="bread soup vegetables pasta cake sauce dough omelette".split(), P="kitchen oven stove bakery restaurant pantry cafeteria grill".split()),
 "sports":   dict(S="striker goalkeeper runner coach captain swimmer referee cyclist".split(), V="scored defended sprinted trained won passed tackled competed".split(),
                  O="goal match race trophy tournament penalty medal season".split(), P="stadium pitch track arena gym pool field court".split()),
 "space":    dict(S="astronaut cosmonaut astronomer physicist commander scientist crew cadet".split(), V="launched orbited observed explored landed docked photographed tracked".split(),
                  O="rocket planet galaxy comet moon satellite asteroid nebula".split(), P="orbit observatory spacecraft launchpad crater cosmos mission telescope".split()),
 "health":   dict(S="doctor nurse patient surgeon pharmacist therapist midwife paramedic".split(), V="treated examined prescribed diagnosed vaccinated healed bandaged operated".split(),
                  O="infection fever medicine injury symptoms wound allergy virus".split(), P="hospital clinic ward pharmacy ambulance surgery laboratory emergency".split()),
 "finance":  dict(S="banker trader investor accountant economist broker auditor shareholder".split(), V="invested borrowed audited saved traded budgeted lent purchased".split(),
                  O="stocks loan interest profit taxes portfolio dividend mortgage".split(), P="bank market exchange treasury brokerage vault boardroom bourse".split()),
 "music":    dict(S="singer guitarist pianist composer drummer violinist conductor bassist".split(), V="performed composed recorded rehearsed strummed sang harmonised improvised".split(),
                  O="melody song album concert symphony chorus rhythm lyrics".split(), P="studio stage theatre orchestra auditorium festival opera club".split()),
 "travel":   dict(S="tourist traveler pilot guide passenger backpacker explorer sightseer".split(), V="booked flew visited boarded wandered packed hiked cruised".split(),
                  O="flight ticket hotel passport suitcase journey itinerary luggage".split(), P="airport station beach resort harbour terminal hostel village".split()),
 "software": dict(S="developer engineer hacker programmer admin tester architect intern".split(), V="coded debugged deployed compiled patched refactored merged scripted".split(),
                  O="software server bug database app code firmware pipeline".split(), P="cloud laptop datacenter repository console network workstation sandbox".split()),
}
FRAMES = ["the {S} {V} the {O}", "the {S} {V} the {O} at the {P}", "at the {P} a {S} {V} some {O}",
          "the {S} {V} near the {P}", "a {O} was {V} by the {S}"]
topic_names = list(POOLS)

def gen_sentence(topic, rnd):
    p = POOLS[topic]
    return rnd.choice(FRAMES).format(S=rnd.choice(p["S"]), V=rnd.choice(p["V"]), O=rnd.choice(p["O"]), P=rnd.choice(p["P"]))

def make_split(per_topic, seed):
    rnd = random.Random(seed); texts, ys = [], []
    for ti, t in enumerate(topic_names):
        for _ in range(per_topic): texts.append(gen_sentence(t, rnd)); ys.append(ti)
    return texts, torch.tensor(ys)

train_s, train_y = make_split(80, seed=1)
doc_s, doc_y = make_split(6, seed=2)
query_s, query_y = make_split(5, seed=3)
print(len(train_s), "train |", len(doc_s), "docs |", len(query_s), "queries |", len(topic_names), "topics")
for i in (0, 85, 170): print(f"  [{topic_names[int(train_y[i])]:8s}] {train_s[i]}")

## 3. Tokens → vectors with hashed sub-words
Test sentences contain words never seen in training. Like FastText (notebook 07) we embed each word as the **mean of its hashed character n-grams**, so unseen words still get meaningful vectors, and no vocabulary file is needed.

In [ ]:
def fnv(s):                                   # deterministic string hash (Python's hash() changes per run)
    h = 2166136261
    for ch in s: h = ((h ^ ord(ch)) * 16777619) & 0xFFFFFFFF
    return h

def words_of(text): return re.findall(r"[a-z0-9]+", text.lower())

_cache = {}
def ngram_ids(word, buckets, nmin=3, nmax=4):
    key = (word, buckets)
    if key not in _cache:
        w = f"<{word}>"
        grams = [w] + [w[i:i + n] for n in range(nmin, nmax + 1) for i in range(len(w) - n + 1)]
        _cache[key] = [fnv(g) % buckets for g in grams]
    return _cache[key]

class SubwordEmbedding(nn.Module):
    """list[list[str]] -> (B,T,d) vectors and (B,T) bool mask. No vocabulary, no <unk>."""
    def __init__(self, d, buckets=4096):
        super().__init__(); self.d, self.buckets = d, buckets
        self.bag = nn.EmbeddingBag(buckets, d, mode="mean"); nn.init.normal_(self.bag.weight, 0, 0.1)
    def forward(self, batch):
        B, T = len(batch), max(len(s) for s in batch)
        flat, offs, pos = [], [], []
        for b, s in enumerate(batch):
            for t, w in enumerate(s):
                offs.append(len(flat)); flat += ngram_ids(w, self.buckets); pos.append(b * T + t)
        vecs = self.bag(torch.tensor(flat), torch.tensor(offs))
        out = torch.zeros(B * T, self.d).index_copy(0, torch.tensor(pos), vecs).view(B, T, self.d)
        mask = torch.zeros(B * T, dtype=torch.bool); mask[torch.tensor(pos)] = True
        return out, mask.view(B, T)

emb = SubwordEmbedding(8); x, m = emb([["hello", "world", "again"], ["short"]])
print(x.shape, m.tolist())

## 4. The sentence encoder
A small Transformer encoder (pre-LN, GELU) followed by a **pooling** step. Padding must be excluded from the pooling, otherwise the sentence vector depends on how much padding the batch happens to have.

In [ ]:
class SentenceEncoder(nn.Module):
    def __init__(self, d=64, out_dim=64, layers=2, heads=4, pooling="mean", max_len=40, buckets=4096, dropout=0.1):
        super().__init__()
        self.pooling, self.cfg = pooling, dict(d=d, out_dim=out_dim, layers=layers, heads=heads, pooling=pooling, max_len=max_len, buckets=buckets, dropout=dropout)
        self.tok = SubwordEmbedding(d, buckets); self.pos = nn.Embedding(max_len, d)
        if pooling == "cls": self.cls = nn.Parameter(torch.zeros(1, 1, d))
        layer = nn.TransformerEncoderLayer(d, heads, dim_feedforward=2 * d, dropout=dropout, activation="gelu", batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.ln = nn.LayerNorm(d); self.proj = nn.Linear(d, out_dim)

    def pool(self, h, mask):
        if self.pooling == "cls": return h[:, 0]
        if self.pooling == "max": return h.masked_fill(~mask.unsqueeze(-1), float("-inf")).amax(1)
        m = mask.unsqueeze(-1).float(); return (h * m).sum(1) / m.sum(1).clamp(min=1)       # masked mean

    def forward(self, batch):                                   # batch: list[list[str]]
        x, mask = self.tok(batch)
        if self.pooling == "cls":
            x = torch.cat([self.cls.expand(len(batch), -1, -1), x], 1)
            mask = torch.cat([torch.ones(len(batch), 1, dtype=torch.bool), mask], 1)
        x = x + self.pos(torch.arange(x.size(1)))
        h = self.ln(self.enc(x, src_key_padding_mask=~mask))
        return F.normalize(self.proj(self.pool(h, mask)), dim=-1)

    @torch.no_grad()
    def encode(self, texts, batch_size=64):                     # sentence-transformers-like API
        was = self.training; self.eval(); outs = []
        for i in range(0, len(texts), batch_size):
            outs.append(self([words_of(t) or ["empty"] for t in texts[i:i + batch_size]]))
        self.train(was); return torch.cat(outs)

enc = SentenceEncoder()
print("parameters:", sum(p.numel() for p in enc.parameters()))
v = enc.encode(train_s[:3]); print(v.shape, "| unit norm:", v.norm(dim=-1))

# padding invariance: a sentence must get the same vector alone or inside a padded batch
enc.eval()
with torch.no_grad():
    a = enc([words_of(train_s[0])]); b = enc([words_of(train_s[0]), words_of(train_s[5] + " " + train_s[6])])[:1]
print("padding invariant:", torch.allclose(a, b, atol=1e-5))

## 5. Contrastive losses
We want matching texts close and non-matching texts far apart. Three standard recipes:

| Loss | Needs | Idea |
|---|---|---|
| **InfoNCE** (in-batch negatives) | pairs `(anchor, positive)` | softmax over the batch; every other item is a negative |
| **Supervised contrastive (SupCon)** | class labels | like InfoNCE, but *all* same-label items are positives |
| **Batch-hard triplet** | class labels | hardest positive vs hardest negative, with a margin |

$$\mathcal L_{\text{InfoNCE}} = -\frac1B\sum_i \log\frac{e^{s(a_i,p_i)/\tau}}{\sum_j e^{s(a_i,p_j)/\tau}}$$

A common bug: if two items in a batch share a topic, InfoNCE treats them as negatives (**false negatives**). SupCon avoids that using labels.

In [ ]:
def info_nce(za, zp, tau=0.1):
    logits = za @ zp.T / tau
    labels = torch.arange(len(za))
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2     # symmetric

def supcon(z, y, tau=0.1):
    sim = z @ z.T / tau
    self_mask = torch.eye(len(z), dtype=torch.bool)
    sim = sim.masked_fill(self_mask, float("-inf"))
    log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    pos = (y[:, None] == y[None, :]) & ~self_mask
    return -(log_prob.masked_fill(~pos, 0).sum(1) / pos.sum(1).clamp(min=1)).mean()

def batch_hard_triplet(z, y, margin=0.3):
    dist = 1 - z @ z.T                                           # cosine distance
    same = y[:, None] == y[None, :]; eye = torch.eye(len(z), dtype=torch.bool)
    hardest_pos = dist.masked_fill(~same | eye, float("-inf")).amax(1)
    hardest_neg = dist.masked_fill(same, float("inf")).amin(1)
    return F.relu(hardest_pos - hardest_neg + margin).mean()

# sanity: a perfect embedding has ~zero loss, a random one does not
y = torch.tensor([0, 0, 1, 1]); perfect = F.normalize(torch.tensor([[1., 0], [1, 0], [0, 1], [0, 1]]) + 1e-3, dim=-1)
rand = F.normalize(torch.randn(4, 2), dim=-1)
print("supcon perfect/random:", supcon(perfect, y).item(), supcon(rand, y).item())
print("triplet perfect/random:", batch_hard_triplet(perfect, y).item(), batch_hard_triplet(rand, y).item())

## 6. Evaluation metrics and a TF-IDF baseline
For each test query we rank all indexed documents and check where the **relevant** ones (same topic) appear.

- **Hit@k** - at least one relevant result in the top-k  
- **Recall@k** - fraction of all relevant items found in the top-k  
- **MRR** - mean of `1 / rank of first relevant result`  
- **nDCG@k** - rewards putting relevant results *early*

In [ ]:
def retrieval_metrics(sim, rel, ks=(1, 3)):
    """sim: (Q,N) similarity, rel: (Q,N) bool relevance."""
    order = sim.argsort(dim=1, descending=True)
    r = rel.gather(1, order).float()                              # relevance in ranked order
    out = {}
    first = (r.cumsum(1) > 0).float().argmax(1) + 1               # rank of first hit (1-based)
    out["MRR"] = (1.0 / first).mean().item()
    for k in ks:
        out[f"Hit@{k}"] = (r[:, :k].sum(1) > 0).float().mean().item()
        out[f"Recall@{k}"] = (r[:, :k].sum(1) / r.sum(1)).mean().item()
    k = max(ks); disc = 1.0 / torch.log2(torch.arange(2, k + 2).float())
    dcg = (r[:, :k] * disc).sum(1); ideal = r.sort(dim=1, descending=True).values[:, :k]
    out[f"nDCG@{k}"] = (dcg / (ideal * disc).sum(1)).mean().item()
    return out

REL = query_y[:, None] == doc_y[None, :]            # (40 queries, 48 docs)

# ---- baseline: TF-IDF cosine, implemented with tensors ----
vocab = sorted({w for s in doc_s for w in words_of(s)}); w2i = {w: i for i, w in enumerate(vocab)}   # TF-IDF indexes the docs
def counts(texts):
    X = torch.zeros(len(texts), len(vocab))
    for i, t in enumerate(texts):
        for w in words_of(t):
            if w in w2i: X[i, w2i[w]] += 1
    return X
df = (counts(doc_s) > 0).sum(0).float(); idf = torch.log((1 + len(doc_s)) / (1 + df)) + 1
tfidf = lambda texts: F.normalize(counts(texts) * idf, dim=-1)
tfidf_metrics = retrieval_metrics(tfidf(query_s) @ tfidf(doc_s).T, REL)
print("TF-IDF baseline        :", {k: round(v, 3) for k, v in tfidf_metrics.items()})

enc0 = SentenceEncoder().eval()
print("untrained encoder      :", {k: round(v, 3) for k, v in retrieval_metrics(enc0.encode(query_s) @ enc0.encode(doc_s).T, REL).items()})

## 7. Training the encoder
Word-dropout is used as light data augmentation. A 150-step run takes a few seconds on CPU.

In [ ]:
def augment(words, p=0.15):
    keep = [w for w in words if random.random() > p]
    return keep if keep else words

train_words = [words_of(s) for s in train_s]
by_topic = collections.defaultdict(list)
for i, t in enumerate(train_y.tolist()): by_topic[t].append(i)

def train_encoder(loss="supcon", pooling="mean", steps=150, seed=0, lr=3e-3, matryoshka=False, out_dim=64, batch=128):
    seed_everything(seed)
    model = SentenceEncoder(pooling=pooling, out_dim=out_dim); opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    model.train()
    for step in range(steps):
        if loss == "infonce":                                   # one pair per topic -> no false negatives in the batch
            pairs = [random.sample(ix, 2) for ix in by_topic.values()]
            za = model([augment(train_words[a]) for a, _ in pairs]); zp = model([augment(train_words[b]) for _, b in pairs])
            l = info_nce(za, zp)
        else:
            idx = random.sample(range(len(train_s)), batch)
            z = model([augment(train_words[i]) for i in idx]); y = train_y[idx]
            fn = supcon if loss == "supcon" else batch_hard_triplet
            if matryoshka:                                      # same loss on nested prefixes 16/32/64 dims
                l = sum(fn(F.normalize(z[:, :k], dim=-1), y) for k in (16, 32, out_dim)) / 3
            else:
                l = fn(z, y)
        opt.zero_grad(); l.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    return model.eval()

def evaluate(model, dims=None):
    q, d = model.encode(query_s), model.encode(doc_s)
    if dims: q, d = F.normalize(q[:, :dims], dim=-1), F.normalize(d[:, :dims], dim=-1)
    return retrieval_metrics(q @ d.T, REL)

t0 = time.time(); model = train_encoder("supcon", "mean", seed=0)
print(f"trained in {time.time() - t0:.1f}s")
print("SupCon + mean pooling:", {k: round(v, 3) for k, v in evaluate(model).items()})
print("TF-IDF baseline      :", {k: round(v, 3) for k, v in tfidf_metrics.items()})

### 7.1 Which loss? Which pooling?
Small-scale results are **noisy**, so we average 3 seeds. Treat the ranking as indicative - differences between losses are usually small on easy data and matter more on large, noisy real datasets.

In [ ]:
def avg_metrics(**kw):
    runs = [evaluate(train_encoder(seed=s, **kw)) for s in range(3)]
    return {k: round(float(np.mean([r[k] for r in runs])), 3) for k in runs[0]}

print(f"{'config':26s} MRR    Hit@1  Recall@3")
for name, kw in [("InfoNCE  / mean", dict(loss="infonce", pooling="mean")),
                 ("SupCon   / mean", dict(loss="supcon", pooling="mean")),
                 ("Triplet  / mean", dict(loss="triplet", pooling="mean")),
                 ("SupCon   / max",  dict(loss="supcon", pooling="max")),
                 ("SupCon   / cls",  dict(loss="supcon", pooling="cls"))]:
    r = avg_metrics(**kw); print(f"{name:26s} {r['MRR']:.3f}  {r['Hit@1']:.3f}  {r['Recall@3']:.3f}")

## 8. Looking at the space
Untrained vs trained: PCA of the 88 held-out sentences (docs + queries) coloured by topic, and the full cosine-similarity matrix sorted by topic.

In [ ]:
all_s = doc_s + query_s; all_y = torch.cat([doc_y, query_y])
def pca2(E):
    X = E - E.mean(0, keepdim=True); _, _, Vh = torch.linalg.svd(X, full_matrices=False); return (X @ Vh[:2].T).numpy()

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5)); cmap = plt.get_cmap("tab10")
for a, (title, m_) in zip(ax[:2], [("untrained", enc0), ("trained (SupCon)", model)]):
    P = pca2(m_.encode(all_s))
    a.scatter(P[:, 0], P[:, 1], c=[cmap(int(i)) for i in all_y], s=35); a.set_title(title)
a = ax[1]; a.legend(handles=[plt.Line2D([], [], marker="o", ls="", c=cmap(i), label=n) for i, n in enumerate(topic_names)], fontsize=6, loc="best")
order = all_y.argsort(); Z = model.encode(all_s)[order]
im = ax[2].imshow(Z @ Z.T, cmap="viridis"); ax[2].set_title("cosine similarity (sorted by topic)"); plt.colorbar(im, ax=ax[2]); plt.tight_layout(); plt.show()

## 9. Semantic search over the corpus
Encode once, search many times. The queries below use word *combinations* that never occur in the docs; the last one deliberately mixes travel and space vocabulary.

In [ ]:
corpus = doc_s
corpus_vecs = model.encode(corpus)
label_of = lambda i: topic_names[int(doc_y[i])]

def search(query, k=3):
    q = model.encode([query]); sims, idx = (q @ corpus_vecs.T)[0].topk(k)
    return [(round(s, 3), label_of(i), corpus[i]) for s, i in zip(sims.tolist(), idx.tolist())]

for q in ["a cook roasted vegetables", "the physicist tracked an asteroid", "the pianist rehearsed a symphony",
          "the developer patched the firmware", "the pilot docked the satellite"]:      # last one mixes travel + space words
    print(f"\nQ: {q}")
    for s, lab, t in search(q): print(f"   {s:.3f}  [{lab:8s}] {t}")

## 10. Matryoshka embeddings - one model, many sizes
Training the loss on **nested prefixes** (first 16, 32, 64 dims) makes the leading dimensions already meaningful, so you can **truncate** vectors to save memory / speed up search without retraining.

In [ ]:
plain = train_encoder("supcon", "mean", seed=0)
mrl = train_encoder("supcon", "mean", seed=0, matryoshka=True)
print(f"{'dims':>5s} | {'plain MRR':>10s} | {'matryoshka MRR':>14s}")
for k in (8, 16, 32, 64):
    print(f"{k:5d} | {evaluate(plain, k)['MRR']:10.3f} | {evaluate(mrl, k)['MRR']:14.3f}")

## 11. Scaling the search
Exact search is one matmul + `topk`: perfect recall, cost `O(N·d)` per query. For millions of vectors we trade a bit of recall for speed:

| Technique | Idea |
|---|---|
| **IVF** | cluster vectors with k-means; search only the `nprobe` nearest clusters |
| **Scalar quantisation (int8)** | 4× smaller, tiny accuracy loss |
| **Binary quantisation** | 32× smaller; compare bits (Hamming), then **re-score** the shortlist with full vectors |
| **HNSW / PQ** | graph / product-quantisation indexes used by FAISS, Qdrant, pgvector |

Below, a synthetic collection of 50 000 unit vectors in 100 clusters.

In [ ]:
seed_everything(0)
N, D, NC = 50_000, 64, 100
centres = F.normalize(torch.randn(NC, D), dim=-1)
vecs = F.normalize(centres[torch.randint(0, NC, (N,))] + 0.09 * torch.randn(N, D), dim=-1)
queries = F.normalize(vecs[torch.randperm(N)[:200]] + 0.05 * torch.randn(200, D), dim=-1)

def exact_topk(q, base, k=10): return (q @ base.T).topk(k, dim=1).indices
t0 = time.perf_counter(); gt = exact_topk(queries, vecs); t_exact = time.perf_counter() - t0
recall = lambda pred: np.mean([len(set(p.tolist()) & set(g.tolist())) / len(g) for p, g in zip(pred, gt)])
print(f"exact search over {N:,} vectors, 200 queries: {t_exact*1000:.0f} ms")

In [ ]:
def kmeans(x, k, iters=15, seed=0):                       # spherical k-means on unit vectors
    g = torch.Generator().manual_seed(seed)
    cent = x[torch.randperm(len(x), generator=g)[:k]].clone()
    for _ in range(iters):
        assign = (x @ cent.T).argmax(1)
        sums = torch.zeros_like(cent).index_add_(0, assign, x); cnt = torch.bincount(assign, minlength=k)
        cent = torch.where(cnt[:, None] > 0, F.normalize(sums, dim=-1), cent)
    return cent, (x @ cent.T).argmax(1)

class IVFIndex:
    def __init__(self, x, nlist=100):
        self.x = x; self.cent, assign = kmeans(x, nlist)
        self.order = assign.argsort(); counts = torch.bincount(assign, minlength=nlist)
        self.offs = torch.cat([torch.zeros(1, dtype=torch.long), counts.cumsum(0)])
    def search(self, q, k=10, nprobe=4):
        out, scanned = [], 0
        for qi in q:
            cl = (self.cent @ qi).topk(nprobe).indices
            cand = torch.cat([self.order[self.offs[c]:self.offs[c + 1]] for c in cl]); scanned += len(cand)
            out.append(cand[(self.x[cand] @ qi).topk(k).indices])
        return torch.stack(out), scanned / len(q)

ivf = IVFIndex(vecs, nlist=100)
print(f"{'nprobe':>6s} | {'recall@10':>9s} | {'avg vectors scanned':>19s} | time")
for nprobe in (1, 2, 4, 8, 16):
    t0 = time.perf_counter(); pred, scanned = ivf.search(queries, nprobe=nprobe); dt = time.perf_counter() - t0
    print(f"{nprobe:6d} | {recall(pred):9.3f} | {scanned:19,.0f} | {dt*1000:.0f} ms")

In [ ]:
# --- int8 scalar quantisation ---
scale = vecs.abs().amax(dim=1, keepdim=True) / 127
q8 = torch.round(vecs / scale).to(torch.int8)                       # stored: N x D bytes
deq = q8.float() * scale
print(f"int8   : {q8.numel()/1e6:.1f} MB vs fp32 {vecs.numel()*4/1e6:.1f} MB | recall@10 = {recall(exact_topk(queries, deq)):.3f}")

# --- binary quantisation + Hamming search + fp32 re-scoring ---
bits = torch.sign(vecs); bits[bits == 0] = 1                          # +-1 ; packed storage = N*D/8 bytes
print(f"binary : {N*D/8/1e6:.2f} MB (32x smaller)")
bq = torch.sign(queries); bq[bq == 0] = 1
ham_sim = bq @ bits.T                                               # = D - 2*Hamming distance
short = ham_sim.topk(100, dim=1).indices                              # cheap shortlist
print("binary only      recall@10 =", round(recall(ham_sim.topk(10, dim=1).indices), 3))
rescored = torch.stack([short[i][(vecs[short[i]] @ queries[i]).topk(10).indices] for i in range(len(queries))])
print("binary + rescore recall@10 =", round(recall(rescored), 3), "(shortlist 100 -> fp32 re-rank)")

## 12. Chunking for RAG
Long documents are split into overlapping **chunks** before embedding: small enough to be specific, large enough to keep context, with overlap so facts at a boundary are not cut.

In [ ]:
def chunk_text(text, size=40, overlap=10):
    words = text.split(); step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, max(1, len(words) - overlap), step)]

doc = " ".join(t for t, y in zip(doc_s, doc_y.tolist()) if topic_names[y] in ("cooking", "space", "software"))
chunks = chunk_text(doc, size=24, overlap=6)
print(len(chunks), "chunks | first:", chunks[0][:100], "...")

chunk_vecs = model.encode(chunks)
q = "the astronomer observed a comet from the observatory"
sims, idx = (model.encode([q]) @ chunk_vecs.T)[0].topk(2)
print("\nQ:", q)
for s, i in zip(sims.tolist(), idx.tolist()): print(f"  {s:.3f} | chunk {i}: {chunks[i][:110]}...")

## 13. Saving and loading an index
Persist the model **config + weights**, the document texts and their vectors; reload and confirm identical search results.

In [ ]:
path = os.path.join(tempfile.gettempdir(), "semantic_index.pt")
torch.save({"config": model.cfg, "state_dict": model.state_dict(), "texts": corpus, "vectors": corpus_vecs}, path)

ckpt = torch.load(path, weights_only=False)                           # contains plain Python lists/dicts we wrote ourselves
loaded = SentenceEncoder(**ckpt["config"]); loaded.load_state_dict(ckpt["state_dict"]); loaded.eval()
q = "the engineer deployed the server"
a = (model.encode([q]) @ corpus_vecs.T)[0].topk(3).indices; b = (loaded.encode([q]) @ ckpt["vectors"].T)[0].topk(3).indices
print("same results after reload:", a.tolist() == b.tolist(), "| file size:", os.path.getsize(path) // 1024, "KB")

## 14. The same pipeline with a pretrained Transformer
In production you rarely train an encoder from scratch: you load a pretrained one (BGE, E5, MiniLM, ...) and reuse **exactly the same** pooling → normalise → cosine → top-k code. This cell is disabled by default so the notebook stays offline; set `RUN_HF = True` after `pip install transformers`.

In [ ]:
RUN_HF = False

def mean_pool(last_hidden, attention_mask):
    m = attention_mask.unsqueeze(-1).float()
    return (last_hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)

class HFEncoder:
    def __init__(self, name="BAAI/bge-small-en-v1.5"):
        from transformers import AutoTokenizer, AutoModel
        self.tok, self.model = AutoTokenizer.from_pretrained(name), AutoModel.from_pretrained(name).eval()
    @torch.no_grad()
    def encode(self, texts, batch_size=32):
        outs = []
        for i in range(0, len(texts), batch_size):
            b = self.tok(texts[i:i + batch_size], padding=True, truncation=True, max_length=256, return_tensors="pt")
            outs.append(F.normalize(mean_pool(self.model(**b).last_hidden_state, b["attention_mask"]), dim=-1))   # BGE uses CLS; mean also works
        return torch.cat(outs)

if RUN_HF:
    hf = HFEncoder()
    print({k: round(v, 3) for k, v in retrieval_metrics(hf.encode(query_s) @ hf.encode(doc_s).T, REL).items()})
else:
    print("RUN_HF is False - skipping download. The class above is a drop-in replacement for `SentenceEncoder.encode`.")

## 15. Choosing and using embeddings - checklist
| Topic | Guidance |
|---|---|
| **Normalise** | L2-normalise, then dot product = cosine; use the same metric for training and search |
| **Query / document asymmetry** | Many models want prefixes such as `query:` / `passage:` - read the model card |
| **Chunk size** | 100-500 tokens with ~10-20 % overlap is a good start; evaluate on *your* data |
| **Evaluate** | Build a small labelled query set; track Recall@k and MRR before and after every change |
| **Hybrid search** | Combine dense scores with BM25 (e.g. reciprocal-rank fusion) - strong on rare keywords, IDs, names |
| **Rerank** | Retrieve top-50 with the bi-encoder, rerank with a cross-encoder (notebook 06) |
| **Cost** | Cache embeddings, batch requests, consider Matryoshka truncation, int8 or binary quantisation |
| **Drift** | Re-embed the whole corpus when you change the model - vectors from different models are not comparable |

## Exercises
1. Add **hard-negative mining** to InfoNCE: for each anchor, add the most similar *wrong-topic* sentence as an extra negative.
2. Implement **reciprocal-rank fusion** of the TF-IDF and dense rankings and compare MRR with each alone.
3. Add a temperature sweep (`tau ∈ {0.03, 0.05, 0.1, 0.2}`) and plot MRR.
4. Build a product-quantisation (PQ) index: split vectors into 8 sub-vectors, k-means each into 256 codes, search with lookup tables.
5. Replace `SentenceEncoder` with `HFEncoder` (`RUN_HF = True`) and compare TF-IDF, your encoder and the pretrained one.
6. Add metadata filtering (e.g. topic or date) to `search`, applied *before* the top-k.

---
### Summary & next step
Work through the exercises above before moving on. Next notebook: **— (embeddings track complete). Continue with LangChain / LangGraph in the next bootcamp module.**.

*Notebook by **Chandan C** · [github.com/chandanc5525](https://github.com/chandanc5525) · Agentic AI BootCamp*